# Preprocessing Dataset — Telco Customer Churn

Notebook ini menjelaskan proses preprocessing secara terstruktur: pemahaman data, audit kualitas, cleaning, transformasi, validasi, visualisasi, dan kesimpulan otomatis.

Dataset yang digunakan adalah data pelanggan telekomunikasi dengan target `Churn`. Identifier pelanggan tidak digunakan sebagai fitur prediktif.

## Tujuan dan Prinsip

Tujuan tugas adalah menghasilkan dataset numerik yang konsisten dan siap untuk analisis atau pemodelan. Setiap perubahan harus memiliki alasan: missing value ditangani berdasarkan tipe data, kategori di-encoding karena tidak memiliki jarak numerik alami, dan standardisasi diterapkan agar fitur numerik memiliki skala sebanding.

Notebook ini mempertahankan dataset mentah di memori sebagai `raw`, sehingga kondisi sebelum dan sesudah dapat dibandingkan.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
sns.set_theme(style='whitegrid')
DATA_PATH = Path('Telco-Customer-Churn.csv')
OUTPUT_PATH = Path('Telco-Customer-Churn-preprocessed.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'File tidak ditemukan: {DATA_PATH.resolve()}')
raw = pd.read_csv(DATA_PATH)
df = raw.copy()
print('Ukuran awal:', raw.shape)
display(raw.head())

## 1. Pemahaman Dataset

Kolom numerik utama adalah `SeniorCitizen`, `tenure`, `MonthlyCharges`, dan `TotalCharges`. Kolom layanan, kontrak, pembayaran, dan demografi bersifat kategorikal. `Churn` adalah target biner dengan label `Yes` dan `No`. `customerID` hanya identifier dan akan dihapus sebelum pemodelan.

In [ ]:
numeric_candidates = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges']
categorical_candidates = [c for c in raw.columns if c not in numeric_candidates + ['customerID','Churn']]
print('Kolom:', list(raw.columns))
print('Numerik:', numeric_candidates)
print('Kategorikal:', categorical_candidates)
print('Distribusi target:')
display(raw['Churn'].value_counts(dropna=False).rename_axis('Churn').to_frame('Jumlah'))

## 2. Audit Kualitas Data

Audit mencakup missing value, blank string, format numerik, duplikasi, dan distribusi target. Blank string diperiksa terpisah karena Pandas tidak selalu membacanya sebagai `NaN`.

In [ ]:
blank = raw.astype('string').apply(lambda c: c.str.strip().eq('').sum())
audit = pd.DataFrame({'dtype':raw.dtypes.astype(str),'missing':raw.isna().sum(),'blank':blank,'unique':raw.nunique(dropna=False)})
print('Duplikasi mentah:', int(raw.duplicated().sum()))
display(audit)
total_numeric = pd.to_numeric(raw['TotalCharges'], errors='coerce')
print('TotalCharges gagal dikonversi:', int(total_numeric.isna().sum() - raw['TotalCharges'].isna().sum()))

## 3. Pemeriksaan Outlier dengan IQR

IQR digunakan sebagai pemeriksaan, bukan alasan otomatis untuk membuang data. Nilai ekstrem dapat merupakan pelanggan nyata, sehingga keputusan penghapusan perlu dukungan konteks domain.

In [ ]:
temp = raw.copy()
temp['TotalCharges'] = pd.to_numeric(temp['TotalCharges'], errors='coerce')
outliers=[]
for col in ['tenure','MonthlyCharges','TotalCharges']:
    q1,q3=temp[col].quantile([.25,.75]); iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    outliers.append({'Kolom':col,'Q1':q1,'Q3':q3,'Batas bawah':lo,'Batas atas':hi,'Jumlah outlier':int(((temp[col]<lo)|(temp[col]>hi)).sum())})
display(pd.DataFrame(outliers))

## 4. Cleaning dan Transformasi

String kosong `TotalCharges` diubah menjadi missing dan diimputasi dengan median. Median dipilih karena lebih tahan terhadap nilai ekstrem. Identifier `customerID` dihapus, duplikasi dibuang setelah ID dihapus, `Churn` diubah menjadi 0/1, fitur kategorikal di-one-hot encode, dan fitur numerik distandardisasi.

Catatan metodologis: untuk pemodelan prediktif, scaler idealnya di-fit setelah train-test split dan hanya pada data training untuk mencegah data leakage.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(r'^\s*$', pd.NA, regex=True), errors='coerce')
missing_before = int(df.isna().sum().sum())
median_total = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_total)
rows_before = len(df)
df = df.drop(columns=['customerID']).drop_duplicates().reset_index(drop=True)
df['Churn'] = df['Churn'].map({'No':0,'Yes':1})
numeric = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges']
categorical = [c for c in df.columns if c not in numeric + ['Churn']]
df[numeric] = StandardScaler().fit_transform(df[numeric])
df = pd.get_dummies(df, columns=categorical, dtype=int)
print('Missing sebelum imputasi:', missing_before)
print('Median TotalCharges:', median_total)
print('Ukuran akhir:', df.shape)

## 5. Perbandingan dan Visualisasi

Perbandingan ini menunjukkan dampak cleaning dan encoding. Jumlah kolom dapat bertambah karena satu kolom kategorikal dipecah menjadi beberapa indikator biner.

In [ ]:
comparison = pd.DataFrame({'Sebelum':[raw.shape[0],raw.shape[1],int(raw.isna().sum().sum()),int(raw.duplicated().sum())],'Sesudah':[df.shape[0],df.shape[1],int(df.isna().sum().sum()),int(df.duplicated().sum())]}, index=['Baris','Kolom','Missing NaN','Duplikasi'])
display(comparison)
fig,ax=plt.subplots(1,2,figsize=(11,4))
raw['Churn'].value_counts().plot.bar(ax=ax[0],title='Target sebelum')
df['Churn'].value_counts().sort_index().plot.bar(ax=ax[1],title='Target sesudah (0=No, 1=Yes)')
plt.tight_layout(); plt.show()

## 6. Validasi dan Penyimpanan

Dataset hanya disimpan setelah seluruh validasi berhasil: tidak ada missing value, tidak ada duplikasi, target tetap biner, dan semua kolom bersifat numerik.

In [ ]:
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0
assert 'Churn' in df.columns
assert set(df['Churn'].unique()).issubset({0,1})
assert all(pd.api.types.is_numeric_dtype(t) for t in df.dtypes)
df.to_csv(OUTPUT_PATH,index=False)
print('VALIDATION: PASS')
print('Tersimpan:', OUTPUT_PATH.resolve())

In [ ]:
# Kesimpulan otomatis berbasis hasil aktual
removed_rows = raw.shape[0] - df.shape[0]
added_cols = df.shape[1] - raw.shape[1]
churn_rate = df['Churn'].mean()
print('='*70)
print('ANALISIS OTOMATIS')
print('='*70)
print(f'- Ukuran: {raw.shape} -> {df.shape}')
print(f'- Baris yang berkurang: {removed_rows}')
print(f'- Kolom yang bertambah karena encoding: {added_cols}')
print(f'- Missing value akhir: {int(df.isna().sum().sum())}')
print(f'- Duplikasi akhir: {int(df.duplicated().sum())}')
print(f'- Proporsi churn: {churn_rate:.2%}')
print('KESIMPULAN: dataset sudah numerik, bersih, tervalidasi, dan siap untuk analisis/modeling.')
if churn_rate < .4 or churn_rate > .6: print('CATATAN: target tidak seimbang; gunakan precision, recall, F1-score, dan confusion matrix.')
print('CATATAN: fit scaler pada data training saja ketika membangun model prediktif.')

## Kesimpulan Laporan

Preprocessing menghasilkan data yang bebas missing value dan duplikasi, memiliki target numerik, fitur kategorikal yang telah di-encoding, serta fitur numerik yang telah distandardisasi. Sel kesimpulan otomatis menyajikan ukuran aktual dan peringatan ketidakseimbangan target setelah notebook dijalankan. Tahap lanjutan adalah train-test split, pipeline preprocessing yang mencegah leakage, pemodelan klasifikasi, dan evaluasi dengan metrik yang sesuai.